# M13 L13: Agents, Tools, and Model Context Protocol (MCP)

**Build a tiny agent runtime: tool registry, MCP-style descriptors, think-act-observe loop.**

Duration: 90 min.  No API keys.

## Plain English (the 12-year-old version)

An "AI agent" sounds like magic, but it is just a loop. Imagine you have a smart friend who can ask you to do things -- look things up, send an email, do math. They keep asking until they have the answer.

That friend is the **LLM**. The loop is: they say "look up the weather in Paris," you do it and tell them the answer, then they say "now send me an email with that weather report," you do it, then they say "done."

**MCP** (Model Context Protocol) is the rulebook for how a friend asks: every "ask" has a name (what to do), a description (what it means), and a JSON recipe for the inputs. The same rulebook works for any friend, so any tool can talk to any agent.

## The 5 things to know

1. **The agent loop is think -> act -> observe**. Same shape every step.
2. **Tools are described with JSON-Schema** (name + description + properties + required).
3. **MCP is the open standard** -- the JSON wire format for tool calls.
4. **Multi-step**: each step's observation feeds the next step's prompt.
5. **max_steps cap**: every agent loop must have a hard limit to prevent infinite loops.

In [1]:
# === Standard setup ===
import sys, importlib.util
from pathlib import Path

def find_root(p):
    while p != p.parent:
        if (p / 'lessons').exists() and (p / 'SYLLABUS.md').exists():
            return p
        p = p.parent
    return Path.cwd()

ROOT = find_root(Path.cwd())
sys.path.insert(0, str(ROOT))

_p = ROOT / 'lessons' / 'm13-agents-mcp' / 'L13-agents-mcp.py'
_spec = importlib.util.spec_from_file_location('L13', _p)
_L13 = importlib.util.module_from_spec(_spec)
sys.modules['L13'] = _L13
_spec.loader.exec_module(_L13)

print(f'Setup OK. Lesson module: {_L13.LESSON_NUMBER}')

Setup OK. Lesson module: M13.L13


In [2]:
# Run the 18 unit tests inline
import inspect
tests = [(n, fn) for n, fn in inspect.getmembers(_L13, inspect.isfunction) if n.startswith('test_')]
print(f'Running {len(tests)} tests...\n')
passed, failed = 0, 0
for name, fn in tests:
    try:
        fn()
        print(f'  PASS  {name}')
        passed += 1
    except AssertionError as e:
        print(f'  FAIL  {name}: {e}')
        failed += 1
print(f'\n{passed}/{len(tests)} tests pass, {failed} fail')
assert failed == 0, f'{failed} test(s) failed -- fix before continuing'

Running 29 tests...

  PASS  test_agent_calculator_task_one_step
  PASS  test_agent_handles_tool_error
  PASS  test_agent_latency_under_budget
  PASS  test_agent_lookup_task_one_step
  PASS  test_agent_math_invariants
  PASS  test_agent_max_steps_stops
  PASS  test_agent_run_final_answer_includes_observation
  PASS  test_agent_run_is_deterministic_same_seed
  PASS  test_agent_run_stops_when_no_tool_matches
  PASS  test_agent_throughput_per_second_inverse_of_latency
  PASS  test_agent_weather_task_one_step
  PASS  test_mcp_schema_helper
  PASS  test_mock_llm_decide_deterministic
  PASS  test_mock_llm_decide_with_seed_kwarg
  PASS  test_mock_llm_picks_calculator
  PASS  test_mock_llm_picks_lookup
  PASS  test_mock_llm_picks_weather
  PASS  test_registry_duplicate_raises
  PASS  test_registry_list_mcp
  PASS  test_registry_list_names_returns_sorted
  PASS  test_registry_register_and_get
  PASS  test_registry_unknown_tool_raises
  PASS  test_sample_tasks_solve
  PASS  test_step_string_trun

In [3]:
# === Section 1: Tool registry + MCP descriptors ===
reg = _L13.build_default_registry()
print(f'Registry size: {len(reg)}\n')
print('MCP-style tool descriptors:')
for t in reg.list_mcp():
    print(f"  - {t['name']:<18} {t['description']}")

Registry size: 5

MCP-style tool descriptors:
  - calculator         Arithmetic: add, subtract, multiply, divide two numbers.
  - search             Search the internal knowledge base for a query string.
  - send_email         Send an email to a recipient with subject and body.
  - get_weather        Get the current weather for a city.
  - lookup_employee    Look up an employee in the HR directory by name.


In [4]:
# === Section 2: MCP schema in detail (calculator) ===
import json
calc_mcp = reg.get('calculator').to_mcp()
print('MCP descriptor for "calculator":')
print(json.dumps(calc_mcp, indent=2))

MCP descriptor for "calculator":
{
  "name": "calculator",
  "description": "Arithmetic: add, subtract, multiply, divide two numbers.",
  "inputSchema": {
    "type": "object",
    "properties": {
      "a": {
        "type": "number",
        "description": "left operand"
      },
      "op": {
        "type": "string",
        "description": "one of + - * /"
      },
      "b": {
        "type": "number",
        "description": "right operand"
      }
    },
    "required": [
      "a",
      "op",
      "b"
    ]
  }
}


In [5]:
# === Section 3: The agent loop -- single task ===
llm = _L13.MockLLM()
agent = _L13.Agent(llm=llm, registry=reg, max_steps=4)
run = agent.run("What is the weather in Paris?")
print(f"task: {run.task}")
print(f"steps: {run.n_steps}   stopped: {run.stopped_reason}\n")
for s in run.steps:
    print(f"  {s}")
print(f"\nfinal: {run.final_answer}")

task: What is the weather in Paris?
steps: 1   stopped: no_tool

  step 0: get_weather({'city': 'Paris'}) -> 'Paris: 72F, sunny'

final: Based on get_weather: Paris: 72F, sunny


In [6]:
# === Section 4: All 3 sample tasks ===
for t in _L13.SAMPLE_TASKS:
    run = agent.run(t)
    print(f"task: {t}")
    print(f"  steps: {run.n_steps}  final: {run.final_answer}")
    print()

task: What is the weather in Paris?
  steps: 1  final: Based on get_weather: Paris: 72F, sunny

task: Calculate 17 * 23 for me.
  steps: 1  final: Based on calculator: 391.0

task: Look up employee Alice Chen in the directory.
  steps: 1  final: Based on lookup_employee: Alice Chen   -- SWE, San Francisco, alice@corp.com



In [7]:
# === Section 5: MockLLM internals ===
print('MockLLM ranking for "What is the weather in Tokyo?":\n')
ranked = llm._rank_tools("What is the weather in Tokyo?", reg)
for name, score in ranked:
    print(f"  {name:<18} score={score}")
print('\nLLM decision:', llm.decide("What is the weather in Tokyo?", [], reg).tool_name)

MockLLM ranking for "What is the weather in Tokyo?":

  get_weather        score=102
  lookup_employee    score=102
  search             score=1
  calculator         score=0
  send_email         score=-50

LLM decision: get_weather


In [8]:
# === Section 6: MCP descriptor for ALL tools (the wire format) ===
print('Full MCP list (what the agent sends the LLM):\n')
import json
all_tools = reg.list_mcp()
print(json.dumps(all_tools, indent=2)[:1500] + '\n  ...(truncated)')

Full MCP list (what the agent sends the LLM):

[
  {
    "name": "calculator",
    "description": "Arithmetic: add, subtract, multiply, divide two numbers.",
    "inputSchema": {
      "type": "object",
      "properties": {
        "a": {
          "type": "number",
          "description": "left operand"
        },
        "op": {
          "type": "string",
          "description": "one of + - * /"
        },
        "b": {
          "type": "number",
          "description": "right operand"
        }
      },
      "required": [
        "a",
        "op",
        "b"
      ]
    }
  },
  {
    "name": "search",
    "description": "Search the internal knowledge base for a query string.",
    "inputSchema": {
      "type": "object",
      "properties": {
        "query": {
          "type": "string",
          "description": "the search query"
        }
      },
      "required": [
        "query"
      ]
    }
  },
  {
    "name": "send_email",
    "description": "Send an email to a

In [9]:
# === Section 7: Error handling ===
# (a) Unknown tool: registry raises KeyError
print('(a) Registry on unknown tool:')
try:
    reg.get('does_not_exist')
except KeyError as e:
    print(f'    caught: {e}')

# (b) Bad args: tool returns ERROR string (no exception)
print('\n(b) Division by zero:')
div_call = llm.decide("What is 5 / 0?", [], reg)
print(f'    call: {div_call.tool_call if False else div_call}')
obs = reg.get(div_call.tool_name).call(**div_call.args)
print(f'    obs: {obs}')

(a) Registry on unknown tool:
    caught: "unknown tool: does_not_exist. available: ['calculator', 'search', 'send_email', 'get_weather', 'lookup_employee']"

(b) Division by zero:
    call: ToolCall(tool_name='calculator', args={'a': 5.0, 'op': '/', 'b': 0.0}, thought="task matches calculator; extracted args {'a': 5.0, 'op': '/', 'b': 0.0}")
    obs: ERROR: division by zero


## Architecture (Mermaid)

```mermaid
flowchart LR
    T[Task] --> A[Agent loop]
    H[History:<br/>past steps] --> A
    A -->|think| LLM[MockLLM:<br/>pick tool + args]
    LLM -->|tool_call| REG[ToolRegistry]
    REG -->|lookup| FN[Tool callable]
    FN -->|observation| A
    A -->|done or max_steps| ANS[Final answer]

    MCP[MCP server:<br/>tools/list] -.JSON-Schema.-> REG
```

**Read as:** the loop is the same every time. The only thing that changes is what tools are in the registry. MCP just standardizes how those tools are described and called.

## Reflect (10 min)

1. Why is JSON-Schema the right format for tool args (vs Python type hints or XML)?
2. The MockLLM uses `_extract_args` to score tools. What happens if two tools extract the same args (e.g., "Bob" matches both `lookup_employee` and a hypothetical `find_by_name`)? How would you break the tie?
3. What is the difference between "stopped: no_tool" and "stopped: max_steps"? Why does it matter?
4. MCP is "JSON-RPC over HTTP/WebSocket." What does JSON-RPC give you that a plain REST API does not? (Hint: look at `tools/list` vs `tools/call`.)
5. (Stretch) Add a `multi_step_agent` that runs two tools in sequence -- first `lookup_employee(Alice)` to get the email, then `send_email(to=alice, body=...)` with a weather report.

---

**Read next:**
- `lessons/m14-data-platforms/` -- data platforms + agentic ETL
- `lessons/m15-staff-design/` -- staff-level system design interview prep
- `SYLLABUS.md` -- the 18 modules and 21-day plan